In [1]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Deep learning libraries

# Statistical distributions for randomized search
from scipy.stats import randint

# Ensure that plots are displayed in the Jupyter Notebook
%matplotlib inline

### Extracting Features and Preparing Data

In [2]:
# The number of seconds to use for each audio clip when extracting features and training the model. Either 3 or 30.
NUMBER_OF_SECONDS = 3
# Columns not taken into consideration for training the model, as they are not relevant for genre classification or are redundant.
COLUMNS_TO_DROP = ['filename', 'label', 'length', 'tempo','harmony_mean', 'harmony_var', 'perceptr_var', 'perceptr_mean']

In [3]:
features = pd.read_csv(f"../data/features_{NUMBER_OF_SECONDS}_sec.csv")

X = features.drop(columns=COLUMNS_TO_DROP)
y = features['label']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)

In [4]:
#LabelEncoder from Scikit-learn is utilized to transform each unique music genre into a specific integer.
labelencoder = LabelEncoder()

y_train = labelencoder.fit_transform(y_train)
y_test = labelencoder.transform(y_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Training K-Nearest Neighbors (KNN) Classifier

In [5]:
# Define the parameter grid for the random search
param_grid = {
    'n_neighbors': randint(1, 15),  # Number of neighbors
    'weights': ['uniform', 'distance'],  # Weight function
    'p': [1, 2]  # Power parameter for the Minkowski distance metric
}

# Create the KNN classifier
knn = KNeighborsClassifier()

# Perform the random search
random_search_knn = RandomizedSearchCV(
    knn, param_distributions=param_grid, n_iter=10, cv=5, random_state=42
)
random_search_knn.fit(X_train, y_train)

# Evaluate the KNN model with the best parameters on the test set
best_knn = random_search_knn.best_estimator_
y_pred_knn = best_knn.predict(X_test)
test_accuracy_knn = accuracy_score(y_test, y_pred_knn)

# Evaluate the KNN model on the training set
y_train_pred_knn = best_knn.predict(X_train)
train_accuracy_knn = accuracy_score(y_train, y_train_pred_knn)

print("Train KNN Accuracy:", train_accuracy_knn)
print("Test KNN Accuracy:", test_accuracy_knn)

Train KNN Accuracy: 0.9992492492492493
Test KNN Accuracy: 0.9074074074074074


In [6]:
import warnings
warnings.filterwarnings('ignore')

from src.extract_features import extract_features

genre = "hiphop"
number = "00000"

# file_path = f'data/genres_original/{genre}/{genre}.{number}.wav'
file_path = f"../data/custom_songs/still_dre.mp3"
song_features = extract_features(file_path, label=genre, segment_duration=NUMBER_OF_SECONDS)
song_features

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,still_dre.mp3,3.000000,0.262101,0.102384,0.009606,0.000067,1642.232538,1.465418e+06,1796.819770,1.379120e+06,...,143.760223,9.649363,58.135048,5.818132,26.628841,0.540584,9.228771,1.289082,11.920897,hiphop
1,still_dre.mp3,3.000000,0.451430,0.077732,0.018593,0.000047,1551.157937,6.557165e+04,1543.813809,3.860327e+04,...,35.483597,12.429777,28.263033,13.377868,17.930885,-3.315775,32.364510,0.832603,17.454857,hiphop
2,still_dre.mp3,3.000000,0.640277,0.051476,0.001674,0.000005,2201.431572,1.092633e+06,2727.306297,4.315915e+05,...,10.922816,7.027939,21.492626,9.135571,35.373390,-1.859190,17.746128,0.176512,7.917391,hiphop
3,still_dre.mp3,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,hiphop
4,still_dre.mp3,3.000000,0.562913,0.057323,0.090052,0.000408,2046.730321,6.483365e+04,2326.990007,1.162601e+04,...,12.624358,-6.036371,14.692680,-2.435097,10.089314,-7.697209,19.641272,-1.153230,24.194496,hiphop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,still_dre.mp3,3.000000,0.334480,0.094347,0.224551,0.017183,2107.048679,1.247725e+06,2288.241115,6.913707e+05,...,48.618118,-2.794625,49.431480,-8.222230,36.895233,-7.348843,51.858234,-0.488915,72.355392,hiphop
93,still_dre.mp3,3.000000,0.309697,0.097903,0.219296,0.018607,2092.680299,1.284557e+06,2251.188882,7.289106e+05,...,66.825829,-1.407784,44.465946,-10.007586,39.812950,-12.797863,76.479439,-4.968388,93.927132,hiphop
94,still_dre.mp3,3.000000,0.319651,0.088920,0.180515,0.006654,2348.392043,1.259370e+06,2494.819297,5.803224e+05,...,43.586716,-2.153838,45.846256,-5.632205,48.643288,-8.194211,36.299347,-3.172640,58.142094,hiphop
95,still_dre.mp3,3.000000,0.319786,0.089155,0.095416,0.003428,1937.377197,8.566807e+05,2282.785734,4.978306e+05,...,29.855093,0.557876,22.092873,-2.786949,36.349663,-4.123679,55.114403,0.340139,52.575939,hiphop


In [7]:
from collections import Counter

# Prepare the song features for prediction
X_song = song_features.drop(columns=COLUMNS_TO_DROP)
X_song_scaled = scaler.transform(X_song)

# Make predictions
y_pred_knn = best_knn.predict(X_song_scaled)

# Map predictions back to genre names
genre_labels = [labelencoder.classes_[int(label)] for label in y_pred_knn]
counter = Counter(genre_labels)

print("Predictions for each segment:", genre_labels)
print("Most common genre:", counter.most_common(1)[0][0])
print("Confidence:", counter.most_common(1)[0][1] / len(genre_labels))

Predictions for each segment: ['classical', 'classical', 'hiphop', 'classical', 'classical', 'rock', 'classical', 'classical', 'reggae', 'rock', 'reggae', 'reggae', 'reggae', 'reggae', 'reggae', 'hiphop', 'hiphop', 'reggae', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'reggae', 'hiphop', 'reggae', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'reggae', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'pop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'pop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'reggae', 'hiphop', 'hiphop', 'reggae', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'reggae', 'reggae', 'reggae', 'hiphop', 'hiphop', 'hiphop', 'hiphop', 'reggae', 'hiphop', 'reggae', 'reggae', 'reggae', 'reggae', 'hiphop', 'hiphop', 'reggae', 'hiphop', '

In [8]:
import joblib

KNN = {
    "model": best_knn,
    "scaler": scaler,
    "label_encoder": labelencoder
}
joblib.dump(KNN, f'../models/KNN_{NUMBER_OF_SECONDS}_sec.joblib')

['../models/KNN_3_sec.joblib']